In [1]:
import os
import json
import re
import numpy as np
from typing import List, Dict, Any
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()


/Users/rayyansiddiqui/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import google.generativeai as genai
import os
import numpy as np
import json
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

In [3]:
class RAGSystem:
    def __init__(self, data_path="rag_data.json"):
        with open(data_path, "r") as f:
            self.data = json.load(f)
        
        self.docs = self.data["documents"]
        self.texts = [doc["text"] for doc in self.docs]
        
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.embed_model.encode(self.texts)
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))

        
    def retrieve(self, query: str, top_k: int = 1):
        query_emb = self.embed_model.encode([query])
        similarities = cosine_similarity(query_emb, self.doc_embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [self.texts[i] for i in top_indices]

    def generate(self, query: str, contexts: list):
        context_text = "\n".join(contexts)
        prompt = f"Answer the question based ONLY on the context provided.\nContext: {context_text}\nQuestion: {query}"
        
        response = self.client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content.strip()

    


In [4]:
import nltk

In [5]:
class RagasEvaluator:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.model = model_name
        genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
        self.embed_model = "gemini-embedding-001"
        
    def _llm_call(self, prompt: str) -> str:
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return response.choices[0].message.content.strip()

    def _parse_json(self, text: str):
        try:
            start_idx = min([text.find(b) for b in ['[', '{'] if text.find(b) != -1])
            end_idx = max([text.rfind(b) for b in [']', '}'] if text.rfind(b) != -1])
            return json.loads(text[start_idx:end_idx+1])
        except: return None

    def calculate_faithfulness(self, question: str, context: str, answer: str) -> dict:
        """Section 3: Faithfulness Logic"""
        # Step 1: Statement Extraction (Paper Prompt)
        extract_prompt = f"""
        Given a question and answer, create one or more statements from each sentence in the given answer.
        question: {question}
        answer: {answer}
        Return ONLY a JSON list of strings.
        """
        statements = self._parse_json(self._llm_call(extract_prompt))
        if not statements: return {"score": 0.0}

        # Step 2: Verification (Paper Prompt)
        # Note: We still use JSON for robustness, but follow the Paper's instruction for 'Yes/No' logic
        verify_prompt = f"""
        Consider the given context and following statements, then determine whether they are supported by the information present in the context. 
        Provide a brief explanation for each statement before arriving at the verdict (Yes/No).
        
        Context: {context}
        Statements: {json.dumps(statements)}
        
        Return ONLY a JSON list of objects: [{{"explanation": "...", "verdict": "Yes/No"}}]
        """
        verdicts = self._parse_json(self._llm_call(verify_prompt))
        if not verdicts: return {"score": 0.0}
        
        supported_count = sum(1 for v in verdicts if v.get("verdict") == "Yes")
        score = float(supported_count / len(statements))
        return {"score": score, "details": verdicts}

    def calculate_answer_relevancy(self, question: str, answer: str, n: int = 3) -> dict:
        """Section 3: Answer Relevance Logic — paper-faithful"""
        # Step 1: Generate n questions from the answer (exact paper prompt)
        gen_questions = []
        for _ in range(n):
            prompt = f"Generate a question for the given answer.\nanswer: {answer}"
            gen_questions.append(self._llm_call(prompt))

        # Step 2: Embed each generated question INDIVIDUALLY (fix for Gemini API)
        orig_res = genai.embed_content(
    model=self.embed_model,
    content=question,
    task_type="retrieval_query"   
)
        orig_emb = np.array(orig_res['embedding']).reshape(1, -1)

        gen_embs = []
        for q in gen_questions:
            res = genai.embed_content(
    model=self.embed_model,
    content=q,
    task_type="retrieval_query"  # same as original, not "retrieval_document"
)
            gen_embs.append(np.array(res['embedding']))

        # Step 3: Mean cosine similarity (Equation 1 in paper)
        similarities = cosine_similarity(orig_emb, gen_embs)[0]
        score = float(np.mean(similarities))
        return {"score": score, "generated_questions": gen_questions}

    def calculate_context_relevancy(self, question: str, context: str) -> dict:
        """Section 3: Context Relevance — paper-faithful"""
        import nltk
        nltk.download('punkt', quiet=True)
        from nltk.tokenize import sent_tokenize

        # Count total sentences using proper sentence tokenizer
        total_sentences = len(sent_tokenize(context))

        # Step 1: Extract relevant sentences using exact paper prompt
        extract_prompt = f"""Please extract relevant sentences from the provided context that \
    can potentially help answer the following question. If no relevant sentences are found, \
    or if you believe the question cannot be answered from the given context, return the phrase \
    "Insufficient Information". While extracting candidate sentences you're not allowed to make \
    any changes to sentences from given context.

    question: {question}
    context: {context}"""

        extracted_raw = self._llm_call(extract_prompt)

        if "Insufficient Information" in extracted_raw:
            return {"score": 0.0, "extracted": []}

        # Count extracted sentences using same tokenizer for consistency
        extracted_sentences = sent_tokenize(extracted_raw)

        # CR = extracted / total (Equation 2 in paper), capped at 1.0
        score = min(float(len(extracted_sentences) / total_sentences), 1.0)
        return {"score": score, "extracted": extracted_sentences}


In [6]:
evaluator = RagasEvaluator() # Using the class we built earlier

with open("paper_tests.json", "r") as f:
    tests = json.load(f)

for test in tests:
    print(f"=== Testing {test['metric']} ===")
    
    if test['metric'] == "Faithfulness":
        # Pass the question, context, and answer as required by the paper's logic
        high = evaluator.calculate_faithfulness(test['question'], test['context'], test['high_answer'])
        low = evaluator.calculate_faithfulness(test['question'], test['context'], test['low_answer'])

        print(f"High Faithfulness Score: {high['score']:.2f}")
        print(f"Low Faithfulness Score:  {low['score']:.2f}")
        
    elif test['metric'] == "Answer Relevancy":
        # Note: You'll need to add calculate_answer_relevancy to your RagasEvaluator class if not there
        high = evaluator.calculate_answer_relevancy(test['question'], test['high_answer'])
        low = evaluator.calculate_answer_relevancy(test['question'], test['low_answer'])
        print(f"High Relevancy Score: {high['score']:.2f}")
        print(f"Low Relevancy Score:  {low['score']:.2f}")
        
    elif test['metric'] == "Context Relevancy":
        high = evaluator.calculate_context_relevancy(test['question'], test['high_context'])
        low = evaluator.calculate_context_relevancy(test['question'], test['low_context'])
        print(f"High Context Relevancy: {high['score']:.2f}")
        print(f"Low Context Relevancy:  {low['score']:.2f}")
    
    print("-" * 30)


=== Testing Faithfulness ===
High Faithfulness Score: 1.00
Low Faithfulness Score:  0.00
------------------------------
=== Testing Answer Relevancy ===


E0000 00:00:1778596271.297868 1568144 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


High Relevancy Score: 0.96
Low Relevancy Score:  0.84
------------------------------
=== Testing Context Relevancy ===
High Context Relevancy: 1.00
Low Context Relevancy:  0.33
------------------------------
=== Testing Answer Correctness ===
------------------------------
